In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner
import os
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

## Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

In [ ]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [4]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'}},
 'required': ['is_professional'],
 'title': 'EmailReview',
 'type': 'object'}

In [5]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Dennis
"""

In [6]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
result = await Runner.run(checker, email)

In [7]:
review = result.final_output
review

EmailReview(is_professional=False)

In [8]:
review.is_professional

False